In [1]:
import pandas as pd
import os
from dotenv import load_dotenv
from supabase import create_client, Client

# 1. Laeme keskkonnamuutujad .env failist
load_dotenv()

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

# Kontrollime, kas muutujad leiti .env failist
if not SUPABASE_URL or not SUPABASE_KEY:
    raise ValueError("⚠️ SUPABASE_URL või SUPABASE_KEY puudub .env failist! Kontrolli .env faili asukohta.")

# 2. Loo ühendus Supabase'iga
supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

# 3. Funktsioon KÕIKIDE ridade laadimiseks (Pagination)
def fetch_all_rows(table_name: str) -> pd.DataFrame:
    """Laadib Supabase tabelist KÕIK read 1000-kaupa Pandas DataFrame'i."""
    all_rows = []
    page_size = 1000
    page = 0
    
    while True:
        start = page * page_size
        end = start + page_size - 1
        
        response = supabase.table(table_name).select("*").range(start, end).execute()
        data = response.data
        
        if not data:
            break
            
        all_rows.extend(data)
        
        if len(data) < page_size:
            break
            
        page += 1
        
    return pd.DataFrame(all_rows)

# 4. Laeme andmed
print("1. Laen andmed tabelist 'sales'...")
df_sales = fetch_all_rows('sales')
print(f"   ✓ sales laetud: {df_sales.shape[0]} rida, {df_sales.shape[1]} veergu")

print("2. Laen andmed tabelist 'customers'...")
df_customers = fetch_all_rows('customers')
print(f"   ✓ customers laetud: {df_customers.shape[0]} rida, {df_customers.shape[1]} veergu")

# 5. Liidame (Merge)
df = pd.merge(df_sales, df_customers, on='customer_id', how='left')

print("\n✅ ANDMED EDUKALT SUPABASE'IST LAETUD JA LIIDETUD!")
print(f"Liidatud DataFrame'i kuju: {df.shape}")
df.head(3)

1. Laen andmed tabelist 'sales'...
   ✓ sales laetud: 10118 rida, 12 veergu
2. Laen andmed tabelist 'customers'...
   ✓ customers laetud: 3150 rida, 9 veergu

✅ ANDMED EDUKALT SUPABASE'IST LAETUD JA LIIDETUD!
Liidatud DataFrame'i kuju: (10118, 20)


,id,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method,first_name,last_name,email,phone,city,registration_date,loyalty_tier,birth_year
0,45703,1,INV-202301-00001,2023-01-10T00:00:00,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart,Hille,Paju,NaN,+372 5429 0294,Tallinn,2022-07-28,bronze,1997.0
1,45704,2,INV-202301-00002,2023-01-16T00:00:00,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks,Merle,Luik,merle.luik@mail.ee,+372 5150 1812,Tallinn,2020-09-22,NaN,1996.0
2,45705,3,INV-202301-00003,2023-01-05T00:00:00,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks,Liina,Saar,liina.saar@gmail.com,+372 8809 7990,Tallinn,2020-03-31,silver,1973.0


In [2]:
print("--- DATA CLEANING ---")

initial_rows = df.shape[0]

# 1. Duplikaatide eemaldamine invoice_id järgi
df_clean = df.drop_duplicates(subset=['invoice_id'], keep='first').copy()
duplicates_removed = initial_rows - df_clean.shape[0]

# 2. Kontrollime ja eemaldame NULL väärtused kriitilistest veergudest
null_counts_before = df_clean[['customer_id', 'sale_date', 'total_price']].isnull().sum()
df_clean = df_clean.dropna(subset=['customer_id', 'sale_date', 'total_price'])

# 3. Parsime kuupäevad datetime tüüpi
df_clean['sale_date'] = pd.to_datetime(df_clean['sale_date'])

# 4. Eemaldame negatiivsed või null-hinnaga outlier'id
df_clean = df_clean[df_clean['total_price'] > 0]

# 5. Puhastusraporti printimine
print("=== PUHASTUSRAPORT ===")
print(f"Algne ridade arv: {initial_rows}")
print(f"Eemaldatud duplikaate (invoice_id): {duplicates_removed}")
print(f"NULL-id kriitilistes veergudes enne eemaldamist:\n{null_counts_before}")
print(f"Lõplik puhastatud ridade arv: {df_clean.shape[0]}")
print(f"Unikaalseid kliente (customer_id): {df_clean['customer_id'].nunique()}")
print(f"Andmete kuupäevavahemik: {df_clean['sale_date'].min().strftime('%Y-%m-%d')} kuni {df_clean['sale_date'].max().strftime('%Y-%m-%d')}")

--- DATA CLEANING ---
=== PUHASTUSRAPORT ===
Algne ridade arv: 10118
Eemaldatud duplikaate (invoice_id): 0
NULL-id kriitilistes veergudes enne eemaldamist:
customer_id    988
sale_date        0
total_price      0
dtype: int64
Lõplik puhastatud ridade arv: 8950
Unikaalseid kliente (customer_id): 2540
Andmete kuupäevavahemik: 2023-01-01 kuni 2026-06-28


In [3]:
print("--- RFM ANALYSIS ---")

# 1. Määrame viitekuupäeva (viimane ostukuupäev andmestikus + 1 päev)
analysis_date = df_clean['sale_date'].max() + pd.Timedelta(days=1)

# 2. Arvutame RFM baasnäitajad iga kliendi kohta
rfm = df_clean.groupby('customer_id').agg({
    'sale_date': lambda x: (analysis_date - x.max()).days, # Recency (päevades)
    'invoice_id': 'nunique',                                # Frequency (unikaalsete ostude arv)
    'total_price': 'sum'                                   # Monetary (kogukulutus EUR)
}).reset_index()

rfm.columns = ['customer_id', 'recency_days', 'frequency', 'monetary_value']

# 3. Arvutame RFM skoorid (1-5) kvintiilide alusel
# Recency: väiksem päevade arv = parem/kõrgem skoor (5)
rfm['R_score'] = pd.qcut(rfm['recency_days'], 5, labels=[5, 4, 3, 2, 1]).astype(int)

# Frequency & Monetary: suurem väärtus = kõrgem skoor (5)
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['M_score'] = pd.qcut(rfm['monetary_value'], 5, labels=[1, 2, 3, 4, 5]).astype(int)

# Summaarne RFM skoor
rfm['RFM_Score'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

# 4. Kliendisegmentide määramine
def assign_segment(score):
    if score >= 13:
        return 'VIP Champions'
    elif score >= 11:
        return 'Loyal Customers'
    elif score >= 9:
        return 'Regular Customers'
    elif score >= 7:
        return 'New / Potential'
    elif score >= 5:
        return 'At Risk'
    else:
        return 'Lost'

rfm['Segment'] = rfm['RFM_Score'].apply(assign_segment)

# 5. Kokkuvõttev tabel
segment_summary = rfm.groupby('Segment').agg(
    Klientide_arv=('customer_id', 'count'),
    Kesk_Recency=('recency_days', 'mean'),
    Kesk_Frequency=('frequency', 'mean'),
    Kogukäive=('monetary_value', 'sum')
).reset_index()

segment_summary['Osakaal_%'] = (segment_summary['Klientide_arv'] / len(rfm) * 100).round(2)
segment_summary = segment_summary.sort_values(by='Kogukäive', ascending=False)

print("=== RFM SEGMENTIDE KOKKUVÕTE ===")
print(segment_summary.to_string(index=False))

--- RFM ANALYSIS ---


=== RFM SEGMENTIDE KOKKUVÕTE ===
          Segment  Klientide_arv  Kesk_Recency  Kesk_Frequency  Kogukäive  Osakaal_%
    VIP Champions            455    534.661538        7.679121 1146295.15      17.91
  Loyal Customers            415    612.946988        4.144578  528261.92      16.34
Regular Customers            512    668.574219        3.177734  474231.56      20.16
  New / Potential            511    701.217221        2.248532  315656.58      20.12
          At Risk            391    772.654731        1.690537  156018.31      15.39
             Lost            256    926.125000        1.167969   56387.02      10.08


In [4]:
import plotly.express as px
import pandas as pd

print("--- JUHENDIKOHASED VISUAALID JA ANALÜÜS ---")

# ---------------------------------------------------------------------
# 1. DIAGRAMM: Segmentide klientide arv (Tulpdiagramm)
# ---------------------------------------------------------------------
fig1 = px.bar(
    segment_summary, 
    x='Segment', 
    y='Klientide_arv', 
    color='Segment',
    text='Klientide_arv',
    title='<b>1. Kliendisegmentide jaotus (Klientide arv)</b>',
    labels={'Klientide_arv': 'Klientide arv', 'Segment': 'Kliendisegment'},
    template='plotly_white'
)
fig1.update_traces(textposition='outside')
fig1.update_layout(showlegend=False, height=400, margin=dict(t=60, b=50))
fig1.show()


# ---------------------------------------------------------------------
# 2. DIAGRAMM: Hajuvusdiagramm (Recency vs Monetary, suurus=Frequency)
# ---------------------------------------------------------------------
# Logaritmiline y-telg tagab, et VIP-id ei suru teisi punkte kokku
fig2 = px.scatter(
    rfm, 
    x='recency_days', 
    y='monetary_value',
    color='Segment',
    size='frequency',
    log_y=True, # Võimaldab näha korraga nii €10 kui €20 000 ostusid
    hover_data=['customer_id'],
    title='<b>2. UrbanStyle Kliendisegmendid (RFM)</b><br><sup>Täpi suurus = ostukordade arv | Y-telg on logaritmilises skaalas</sup>',
    labels={
        'recency_days': 'Päevi viimasest ostust (Recency)', 
        'monetary_value': 'Kogukulutus EUR (Monetary)',
        'Segment': 'Segment'
    },
    template='plotly_white',
    opacity=0.7
)
fig2.update_layout(height=450, margin=dict(t=60, b=50))
fig2.show()


# ---------------------------------------------------------------------
# 3. DIAGRAMM: Top 10 VIP klienti kogukulutuse järgi
# ---------------------------------------------------------------------
top_vip = rfm[rfm['Segment'] == 'VIP Champions'].nlargest(10, 'monetary_value')

fig3 = px.bar(
    top_vip,
    x='customer_id',
    y='monetary_value',
    text='monetary_value',
    title='<b>3. Top 10 VIP Klienti Kogukulutuse Järgi</b>',
    labels={'monetary_value': 'Kogukulutus (€)', 'customer_id': 'Kliendi ID'},
    template='plotly_white',
    color_discrete_sequence=['#636EFA']
)
fig3.update_traces(texttemplate='€%{text:,.0f}', textposition='outside')
fig3.update_layout(yaxis_tickprefix='€', height=400, margin=dict(t=60, b=50))
fig3.show()

--- JUHENDIKOHASED VISUAALID JA ANALÜÜS ---
